# POF Forecast

Importing all the necessary python libraries

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
import cftime
from xarray.coding.times import CFDatetimeCoder
import os


Configuration Options

Here we define the paths for files and the dates we will use for our forecast.

We also load the model we created in the Trainer notebook


In [2]:
# Folder structure
base_path = Path("./data/")
os.makedirs("./outputs/", exist_ok=True)

# Dates of Forecast
year = 2003
month = 1

# Where we will store our predictions
output_file = f"./outputs/POF_prediction_{year}_{month:02d}.nc"

# Load trained model
model = joblib.load("./data/POF_model.joblib")


Load Static Data

We will load some CLIMATE data files

In [3]:
# Open the Climate files
time_coder = CFDatetimeCoder(use_cftime=True)
PO = xr.open_dataset(base_path / "CLIMATE/POP_2020.nc")
RD = xr.open_dataset(base_path / "CLIMATE/road_density_2015_agg_r.nc")

# Extract the arrays
PO_arr = PO.population_density.values  
RD_arr = RD.road_length.values         

Load Dynamic Data

Here we open the data files we retrieved from the XDS and the CDS as well as the Active Fire Maps

We iterate through each day within the dataset and store the into an output NetCDF file.

In [4]:

# Define the paths of the files
ds_paths = {
    "AF": f"ACTIVE_FIRE_MAP_{year}_{month:02d}_R.nc",
    "FU": f"FUEL_MAP_{year}_{month:02d}_R.nc",
    "DF": f"DFMC_MAP_{year}_{month:02d}_R.nc",
    "LF": f"LFMC_MAP_{year}_{month:02d}_R.nc",
    "PR": f"P_{year}_{month:02d}.nc",
    "T2": f"T2M_{year}_{month:02d}.nc",
    "RH": f"RH_{year}_{month:02d}.nc",
    "WS": f"WS_{year}_{month:02d}.nc",
}
# Open all dynamic datasets
with xr.open_dataset(base_path / ds_paths["FU"]) as FU, \
     xr.open_dataset(base_path / ds_paths["DF"]) as DF, \
     xr.open_dataset(base_path / ds_paths["LF"]) as LF, \
     xr.open_dataset(base_path / ds_paths["PR"]) as PR, \
     xr.open_dataset(base_path / ds_paths["T2"]) as T2, \
     xr.open_dataset(base_path / ds_paths["RH"]) as RH, \
     xr.open_dataset(base_path / ds_paths["WS"]) as WS, \
     xr.open_dataset(base_path / ds_paths["AF"]) as AF:

    n_days = len(AF.ACTIVE_FIRE)
    all_grids = []

    for i in range(n_days):
        # Extract arrays for timestep i
        FU_LL = FU.Live_Leaf[i].values
        FU_LW = FU.Live_Wood[i].values
        FU_DF = FU.Dead_Foliage[i].values
        FU_DW = FU.Dead_Wood[i].values
        DF_ = DF.DFMC_Foliage[i].values
        DW_ = DF.DFMC_Wood[i].values
        LF_ = LF.LFMC[i].values
        PR_ = PR.tp[i].values
        T2_ = T2.t2m[i].values
        RH_ = RH.rh[i].values
        WS_ = WS.ws[i].values

        # Mask where total fuel > 0
        ft = FU_LL + FU_LW + FU_DF + FU_DW
        mask = ft > 0

        # Flatten and build dataframe for prediction
        feature_arrays = {
            "PR": PR_[mask],
            "T2": T2_[mask],
            "RH": RH_[mask],
            "WS": WS_[mask],
            "FU_LL": FU_LL[mask],
            "FU_LW": FU_LW[mask],
            "FU_DF": FU_DF[mask],
            "FU_DW": FU_DW[mask],
            "DF": DF_[mask],
            "DW": DW_[mask],
            "LF": LF_[mask],
            "PO": PO_arr[mask],
            "RD": RD_arr[mask],
        }
        X_pred = pd.DataFrame(feature_arrays)

        # Predict probability
        y_proba = model.predict_proba(X_pred)[:, 1]

        # Create full grid and fill masked values
        fire_prob_grid = np.full(ft.shape, np.nan, dtype=float)
        fire_prob_grid[mask] = y_proba
        all_grids.append(fire_prob_grid)

    # Stack all timesteps into 3D array (time, lat, lon)
    fire_prob_array = np.stack(all_grids, axis=0)

    # -----------------------------
    # SAVE TO NETCDF
    # -----------------------------
    time = [cftime.DatetimeJulian(year, month, day+1) for day in range(n_days)]
    ds_out = xr.Dataset(
        {"fire_probability": (["time", "lat", "lon"], fire_prob_array)},
        coords={
            "time": time,
            "latitude": PO.latitude,
            "longitude": PO.longitude
        }
    )

    ds_out.to_netcdf(output_file)
    print(f"✅ Prediction saved → {output_file}")

✅ Prediction saved → ./outputs/POF_prediction_2003_01.nc
